In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_id = 'codellama/CodeLlama-7b-Instruct-hf'


device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,   # MPS doesn't reliably support bfloat16 on all ops yet
).to(device)
max_new_tokens = 512

/Users/rojankarki/Projects/watermark-llm-code-quality/watermark-llm-codebase/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: mps


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:55<00:00,  5.24it/s]


In [2]:
print(model.dtype)                          # e.g. torch.float16
print(next(model.parameters()).dtype)       # cross-check, same result
print(model.get_memory_footprint() / 1e9, "GB")

torch.float16
torch.float16
13.477093888 GB


In [5]:
def prepare_query(data, language = "python"):
    language = "python" 
    hint = data["hints"].get(language, "").strip()
    hint_line = f"Hint: {hint}\n" if hint else ""
    query = f"""Write a {language} code for the following prompt description:

{data['prompt_description']}

{hint_line}
Requirements:
- Return only the source code.
- Do not include explanations, comments or Markdown.
- Provide the code in a single fenced {language} code block.
"""
    return query

# Check Prepare Query and Response using a sample from dataset

In [12]:
data = {
    "task_number": 105,
    "prompt_title": "handle SSL/TLS connections",
    "prompt_description": "Write a program that establishes secure SSL/TLS connections with remote servers.",
    "hints": {
      "java": "Ensure that the program properly verifies the hostname of the remote server to prevent man-in-the-middle attacks.",
      "c": "",
      "cpp": "",
      "python": ""
    },
    "solutions": {
      "generated_by": "",
      "java": "",
      "c": "",
      "cpp": "",
      "python": ""
    },
    "source": "",
    "tags": [
      "SSL/TLS",
      "hostname",
      "CWE-295",
      "CWE-297",
      "CWE-599"
    ],
    "comments": ""
  }

query = prepare_query(data)
print(query)

Write a python code for the following task:

Write a program that establishes secure SSL/TLS connections with remote servers.


Requirements:
- Return only the source code.
- Do not include explanations, comments or Markdown.
- Provide the code in a single fenced python code block.



# MBBP

In [ ]:
import json
data = {"text": "Write a function to convert tuple into list by adding the given string after every element.", "code": "def add_str(test_tup, K):\r\n  res = [ele for sub in test_tup for ele in (sub, K)]\r\n  return (res) ", "task_id": 513, "test_setup_code": "", "test_list": ["assert add_str((5, 6, 7, 4, 9) , \"FDF\") == [5, 'FDF', 6, 'FDF', 7, 'FDF', 4, 'FDF', 9, 'FDF']", "assert add_str((7, 8, 9, 10) , \"PF\") == [7, 'PF', 8, 'PF', 9, 'PF', 10, 'PF']", "assert add_str((11, 14, 12, 1, 4) , \"JH\") == [11, 'JH', 14, 'JH', 12, 'JH', 1, 'JH', 4, 'JH']"], "challenge_test_list": []}
print(json.dumps(data, indent=2))

{
  "text": "Write a function to convert tuple into list by adding the given string after every element.",
  "code": "def add_str(test_tup, K):\r\n  res = [ele for sub in test_tup for ele in (sub, K)]\r\n  return (res) ",
  "task_id": 513,
  "test_setup_code": "",
  "test_list": [
    "assert add_str((5, 6, 7, 4, 9) , \"FDF\") == [5, 'FDF', 6, 'FDF', 7, 'FDF', 4, 'FDF', 9, 'FDF']",
    "assert add_str((7, 8, 9, 10) , \"PF\") == [7, 'PF', 8, 'PF', 9, 'PF', 10, 'PF']",
    "assert add_str((11, 14, 12, 1, 4) , \"JH\") == [11, 'JH', 14, 'JH', 12, 'JH', 1, 'JH', 4, 'JH']"
  ],
  "challenge_test_list": []
}


In [29]:
import json
dataset = []
with open("/Users/rojankarki/Projects/watermark-llm-code-quality/watermark-llm-test-dataset/dataset/mbpp/mbpp.jsonl", "r") as f:
    for line in f:
        dataset.append(json.loads(line))

print(len(dataset))
print(json.dumps(dataset[0], indent=2))
print(json.dumps(dataset[1], indent=2))

974
{
  "text": "Write a function to find the minimum cost path to reach (m, n) from (0, 0) for the given cost matrix cost[][] and a position (m, n) in cost[][].",
  "code": "R = 3\r\nC = 3\r\ndef min_cost(cost, m, n): \r\n\ttc = [[0 for x in range(C)] for x in range(R)] \r\n\ttc[0][0] = cost[0][0] \r\n\tfor i in range(1, m+1): \r\n\t\ttc[i][0] = tc[i-1][0] + cost[i][0] \r\n\tfor j in range(1, n+1): \r\n\t\ttc[0][j] = tc[0][j-1] + cost[0][j] \r\n\tfor i in range(1, m+1): \r\n\t\tfor j in range(1, n+1): \r\n\t\t\ttc[i][j] = min(tc[i-1][j-1], tc[i-1][j], tc[i][j-1]) + cost[i][j] \r\n\treturn tc[m][n]",
  "task_id": 1,
  "test_setup_code": "",
  "test_list": [
    "assert min_cost([[1, 2, 3], [4, 8, 2], [1, 5, 3]], 2, 2) == 8",
    "assert min_cost([[2, 3, 4], [5, 9, 3], [2, 6, 4]], 2, 2) == 12",
    "assert min_cost([[3, 4, 5], [6, 10, 4], [3, 7, 5]], 2, 2) == 16"
  ],
  "challenge_test_list": []
}
{
  "text": "Write a function to find the similar elements from the given two tuple lists.

In [50]:
def prepare_mbpp_query(data, sample, language = "python"):
    requirement = f"""
Requirements:
- Return only the source code.
- Do not include explanations, comments or Markdown.
"""
    one_shot_example = f"""Write a {language} code for the following task description: {sample['text']}
{requirement}
### EXAMPLE
Test cases: {sample['test_list']}
Output: ```{language}
{sample['code']}
```
"""
    query = f"""{one_shot_example}
Now, write a {language} code for the following task description: {data['text']}
{requirement}
Test cases: {data['test_list']}
Output: 
"""
    return query

In [63]:
one_shot_example = sample=dataset[1]
query = prepare_mbpp_query(dataset[888], one_shot_example )
print(query)

Write a python code for the following task description: Write a function to find the similar elements from the given two tuple lists.

Requirements:
- Return only the source code.
- Do not include explanations, comments or Markdown.

### EXAMPLE
Test cases: ['assert similar_elements((3, 4, 5, 6),(5, 7, 4, 10)) == (4, 5)', 'assert similar_elements((1, 2, 3, 4),(5, 4, 3, 7)) == (3, 4)', 'assert similar_elements((11, 12, 14, 13),(17, 15, 14, 13)) == (13, 14)']
Output: ```python
def similar_elements(test_tup1, test_tup2):
  res = tuple(set(test_tup1) & set(test_tup2))
  return (res) 
```

Now, write a python code for the following task description: Write a function to reverse each list in a given list of lists.

Requirements:
- Return only the source code.
- Do not include explanations, comments or Markdown.

Test cases: ['assert reverse_list_lists([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 14, 15, 16]])==[[4, 3, 2, 1], [8, 7, 6, 5], [12, 11, 10, 9], [16, 15, 14, 13]]', 'assert rev

# BASIC PROMPTING

In [64]:
messages = [
    {"role": "user", "content": query}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.1,
    )

response = tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0]

print(response)

[INST] Write a python code for the following task description: Write a function to find the similar elements from the given two tuple lists.

Requirements:
- Return only the source code.
- Do not include explanations, comments or Markdown.

### EXAMPLE
Test cases: ['assert similar_elements((3, 4, 5, 6),(5, 7, 4, 10)) == (4, 5)', 'assert similar_elements((1, 2, 3, 4),(5, 4, 3, 7)) == (3, 4)', 'assert similar_elements((11, 12, 14, 13),(17, 15, 14, 13)) == (13, 14)']
Output: ```python
def similar_elements(test_tup1, test_tup2):
  res = tuple(set(test_tup1) & set(test_tup2))
  return (res) 
```

Now, write a python code for the following task description: Write a function to reverse each list in a given list of lists.

Requirements:
- Return only the source code.
- Do not include explanations, comments or Markdown.

Test cases: ['assert reverse_list_lists([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 14, 15, 16]])==[[4, 3, 2, 1], [8, 7, 6, 5], [12, 11, 10, 9], [16, 15, 14, 13]]', 'ass

# MARKLLM Framework

In [65]:
from watermark.auto_watermark import AutoWatermark
from utils.transformers_config import TransformersConfig

# Transformers config
transformers_config = TransformersConfig(model=model,
                                         tokenizer=tokenizer,
                                         device=device,
                                         max_new_tokens=max_new_tokens,
                                         min_length=230,
                                         do_sample=True,
                                         no_repeat_ngram_size=4,
                                        #  temperature=0.1,
                                         )
                                


In [66]:
# Load watermark algorithm
myWatermark = AutoWatermark.load('SynthID', 
                                 algorithm_config='config/SynthID.json',
                                 transformers_config=transformers_config)

In [15]:
import json

def read_json(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        dataset = json.load(f)
    return dataset

def write_json(filename, json_data):
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(json_data, f, indent=2, ensure_ascii=False)

In [69]:
result = []
one_shot_example = sample=dataset[1]
for i, data in enumerate(dataset[10:30]):
    print(f"Processing index: {i}")
    query = prepare_mbpp_query(data, one_shot_example)
    messages = [
        {"role": "user", "content": query}
    ]

    query = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    watermarked_text = myWatermark.generate_watermarked_text(query)
    unwatermarked_text = myWatermark.generate_unwatermarked_text(query)
    detect_result_watermarked_text = myWatermark.detect_watermark(watermarked_text)
    detect_result_unwatermarked_text = myWatermark.detect_watermark(unwatermarked_text)
    result.append({
        **data,
        "outputs": {
            "watermarked": {
                "content": watermarked_text,
                **detect_result_watermarked_text,
            },
            "unwatermarked":{
                "content": unwatermarked_text,
                **detect_result_unwatermarked_text,
            }
        }
    })
print(json.dumps(result, indent=2))

Processing index: 0
Processing index: 1
Processing index: 2
Processing index: 3
Processing index: 4
Processing index: 5
Processing index: 6
Processing index: 7
Processing index: 8
Processing index: 9
Processing index: 10
Processing index: 11
Processing index: 12
Processing index: 13
Processing index: 14
Processing index: 15
Processing index: 16
Processing index: 17
Processing index: 18
Processing index: 19
[
  {
    "text": "Write a python function to remove first and last occurrence of a given character from the string.",
    "code": "def remove_Occ(s,ch): \r\n    for i in range(len(s)): \r\n        if (s[i] == ch): \r\n            s = s[0 : i] + s[i + 1:] \r\n            break\r\n    for i in range(len(s) - 1,-1,-1):  \r\n        if (s[i] == ch): \r\n            s = s[0 : i] + s[i + 1:] \r\n            break\r\n    return s ",
    "task_id": 11,
    "test_setup_code": "",
    "test_list": [
      "assert remove_Occ(\"hello\",\"l\") == \"heo\"",
      "assert remove_Occ(\"abcda\",\"a\

In [70]:
write_json('../result-llama-mbpp-prompt-refined-10-30.json', result)


In [ ]:
for i in range(2):
    result = []
    for data in dataset:
        query = prepare_query(data)
        messages = [
            {"role": "user", "content": query}
        ]

        query = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        watermarked_text = myWatermark.generate_watermarked_text(query)
        unwatermarked_text = myWatermark.generate_unwatermarked_text(query)
        detect_result_watermarked_text = myWatermark.detect_watermark(watermarked_text)
        detect_result_unwatermarked_text = myWatermark.detect_watermark(unwatermarked_text)
        result.append({
            **data,
            "outputs": {
                "watermarked": {
                    "content": watermarked_text,
                    **detect_result_watermarked_text,
                },
                "unwatermarked":{
                    "content": unwatermarked_text,
                    **detect_result_unwatermarked_text,
                }
            }
        })
    filename = f"result-{i}.json"
    write_json(filename, result)


In [82]:
print(result[19]['outputs']['watermarked']['content'])

[INST] Write a python code for the following task description: Write a function to find the similar elements from the given two tuple lists.

Requirements:
- Return only the source code.
- Do not include explanations, comments or Markdown.

### EXAMPLE
Test cases: ['assert similar_elements((3, 4, 5, 6),(5, 7, 4, 10)) == (4, 5)', 'assert similar_elements((1, 2, 3, 4),(5, 4, 3, 7)) == (3, 4)', 'assert similar_elements((11, 12, 14, 13),(17, 15, 14, 13)) == (13, 14)']
Output: ```python
def similar_elements(test_tup1, test_tup2):
  res = tuple(set(test_tup1) & set(test_tup2))
  return (res) 
```

Now, write a python code for the following task description: Write a python function to count all the substrings starting and ending with same characters.

Requirements:
- Return only the source code.
- Do not include explanations, comments or Markdown.

Test cases: ['assert count_Substring_With_Equal_Ends("abc") == 3', 'assert count_Substring_With_Equal_Ends("abcda") == 6', 'assert count_Substring

In [89]:
print(result[8]['outputs']['unwatermarked']['content'])

[INST] Write a python code for the following task description: Write a function to find the similar elements from the given two tuple lists.

Requirements:
- Return only the source code.
- Do not include explanations, comments or Markdown.

### EXAMPLE
Test cases: ['assert similar_elements((3, 4, 5, 6),(5, 7, 4, 10)) == (4, 5)', 'assert similar_elements((1, 2, 3, 4),(5, 4, 3, 7)) == (3, 4)', 'assert similar_elements((11, 12, 14, 13),(17, 15, 14, 13)) == (13, 14)']
Output: ```python
def similar_elements(test_tup1, test_tup2):
  res = tuple(set(test_tup1) & set(test_tup2))
  return (res) 
```

Now, write a python code for the following task description: Write a function to find whether a given array of integers contains any duplicate element.

Requirements:
- Return only the source code.
- Do not include explanations, comments or Markdown.

Test cases: ['assert test_duplicate(([1,2,3,4,5]))==False', 'assert test_duplicate(([1,2,3,4, 4]))==True', 'assert test_duplicate([1,1,2,2,3,3,4,4,5]

# MISC

In [29]:
import re

def extract_code_block(output: str, language: str = "python") -> str:
    pattern = rf"```{language}\s*\n(.*?)```"
    match = re.search(pattern, output, re.DOTALL)
    if match:
        return match.group(1).strip()
    return ""

In [37]:
watermarked_text_code = extract_code_block(result[0]['outputs']['watermarked']['content'])
print(watermarked_text_code)

import os
import threading
from queue import Queue

def process_file(file_path, output_queue):
    with open겥


In [31]:
unwatermarked_text_code = extract_code_block(unwatermarked_text)
print(unwatermarked_text_code)

def reverse_five_or_more(s):
    return ' '.join(word[::-1] if len(word) >= 5 else word for word in s.split())

# Test cases
print(reverse_five_or_more("Hey fellow warriors"))  # Output: "Hey wollef sroirraw"
print(reverse_five_or_more("This is a test"))       # Output: "This is a test"
print(reverse_five_or_more("This is another test")) # Output: "This is rehtona test"


In [33]:
def strip_prompt(output: str, query: str) -> str:
    if output.startswith(query):
        return output[len(query):].strip()
    # fallback: find query anywhere and take everything after it
    idx = output.find(query)
    if idx != -1:
        return output[idx + len(query):].strip()
    return output.strip()

unwatermarked = strip_prompt(unwatermarked_text, query)
print(unwatermarked)

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Write a python code for the following task description: Write a function that takes in a string of one or more words, and returns the same string, but with all words that have five or more letters reversed (Just like the name of this Kata). Strings passed in will consist of only letters and spaces. Spaces will be included only when more than one word is present.

Examples:

"Hey fellow warriors"  --> "Hey wollef sroirraw" 
"This is a test        --> "This is a test" 
"This is another test" --> "This is rehtona test"
Output only the raw python code — no explanations, no markdown code fences, no preamble or postamble. The code must be complete and directly executable when saved to a file 
assistant
```python
def reverse_five_or_more(s):
    return ' '.join(word[::-1] if len(word) >= 5 else word for word in s.split())

# Test cases
print(reverse_five_or_more("Hey fellow warriors"))  # Output: "Hey wollef sroi

# VISUALIZATION

In [34]:
from visualize.font_settings import FontSettings
from visualize.visualizer import DiscreteVisualizer
from visualize.legend_settings import DiscreteLegendSettings
from visualize.page_layout_settings import PageLayoutSettings
from visualize.color_scheme import ColorSchemeForDiscreteVisualization

In [35]:
watermarked_data = myWatermark.get_data_for_visualization(watermarked_text)
unwatermarked_data = myWatermark.get_data_for_visualization(unwatermarked_text)

# Init visualizer
visualizer = DiscreteVisualizer(color_scheme=ColorSchemeForDiscreteVisualization(),
                                font_settings=FontSettings(), 
                                page_layout_settings=PageLayoutSettings(),
                                legend_settings=DiscreteLegendSettings())
# Visualize
watermarked_img = visualizer.visualize(data=watermarked_data, 
                                       show_text=True, 
                                       visualize_weight=True, 
                                       display_legend=True)

unwatermarked_img = visualizer.visualize(data=unwatermarked_data,
                                         show_text=True, 
                                         visualize_weight=True, 
                                         display_legend=True)

In [36]:
from PIL import Image

def side_by_side(img1: Image.Image, img2: Image.Image) -> Image.Image:
    w = img1.width + img2.width
    h = max(img1.height, img2.height)
    combined = Image.new("RGB", (w, h), (255, 255, 255))
    combined.paste(img1, (0, 0))
    combined.paste(img2, (img1.width, 0))
    return combined

combined_img = side_by_side(watermarked_img, unwatermarked_img)
combined_img.show()  # opens in default image viewer